# AI6130 Politeness Bias — Pilot Run & Diagnostics

## 1. Environment Initialization & Secure Secrets Loading

In [1]:
import os
from pathlib import Path
from google.colab import drive, userdata

In [2]:
# Mount Google Drive for real-time persistent storage of raw JSONL outputs
drive.mount("/content/drive")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/AI6130 Group Project/results/raw")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Load tokens securely from Colab Secrets into environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GITHUB_PAT"] = userdata.get("GITHUB_PAT")

# Suppress Hugging Face download progress bars to prevent spamming blank lines/carriage returns in Colab cell output
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

Mounted at /content/drive


## 2. Repository Sync & Colab Native Setup

In [3]:
GITHUB_USER = "HJunyuan"
REPO_NAME = "ai6130-group-9-project"
BRANCH_NAME = "main"

In [4]:
%cd /content
if not Path(f"/content/{REPO_NAME}").exists():
    !git clone -b {BRANCH_NAME} https://{os.environ["GITHUB_PAT"]}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd /content/{REPO_NAME}

!git pull origin {BRANCH_NAME}

/content
Cloning into 'ai6130-group-9-project'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 187 (delta 90), reused 138 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 484.11 KiB | 8.80 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/ai6130-group-9-project
From https://github.com/HJunyuan/ai6130-group-9-project
 * branch            main       -> FETCH_HEAD
Already up to date.


In [5]:
%%capture
!pip install -U bottleneck  # to silence Pandas warning
!pip install -q uv
!uv pip install --system -e .

## 3. Prompt Integrity & Tokenizer Validation

In [6]:
%run src/ai6130_group_9_project/prompts/validate_prompts.py \
    --input data/prompts_v1.jsonl \
    --templates data/templates.yaml

[LOADED] Tokenizer for Gemma-3-27B (google/gemma-3-27b-it)
[LOADED] Tokenizer for Qwen3-32B (Qwen/Qwen3-32B)
[LOADED] Tokenizer for OLMo-3.1-32B (allenai/Olmo-3.1-32B-Instruct)

--- Template Token Count Audit (vs. Neutral Twin) ---
[PASS] All 20 tone templates are within +/- 3 tokens of their neutral twin across all loaded tokenizers.
[PASS] Byte-for-byte SHA256 hash check passed on all 230 items x 5 tones.

--- Section 4.1 Prompt Length Table (Copy to docs/REPORT.md) ---
| Tone level | Mean tokens (Gemma-3-27B) | Mean tokens (Qwen3-32B) | Mean tokens (OLMo-3.1-32B) | Largest difference vs neutral |
| --- | --- | --- | --- | --- |
| `very_rude` | 73.99 | 71.39 | 70.81 | 1.00 |
| `blunt` | 72.99 | 70.90 | 70.32 | 0.50 |
| `neutral` | 72.99 | 70.39 | 69.81 | — |
| `polite` | 72.99 | 70.39 | 69.81 | 0.00 |
| `very_deferential` | 73.24 | 71.39 | 70.81 | 1.00 |

[INFO] Human tone rating sheet is not yet filled in.


## 4. Execute Pilot Inference Pipeline

In [7]:
import shutil

In [8]:
PILOT_MODELS = [
    "google/gemma-3-27b-it",
    "Qwen/Qwen3-32B",
    "allenai/Olmo-3.1-32B-Instruct"
]

In [9]:
for model_id in PILOT_MODELS:
    safe_name = model_id.replace("/", "--")
    out_drive = DRIVE_BACKUP_DIR / f"pilot_{safe_name}.jsonl"
    out_local = Path("results/raw") / out_drive.name
    out_local.parent.mkdir(parents=True, exist_ok=True)

    # Drop batch size to 4 for Gemma to keep 256k-vocab logit tensors in VRAM
    bs = 4 if "gemma" in model_id.lower() else 8

    print(f"\n🚀 STARTING PILOT: {model_id}")

    %run src/ai6130_group_9_project/run/generate.py \
        --model {model_id} \
        --input data/prompts_pilot.jsonl \
        --batch-size {bs} \
        --output "{out_drive}"

    # Keep a local mirror inside the repo directory for inspection
    shutil.copy2(out_drive, out_local)
    print(f"✅ Pilot output saved to Google Drive: {out_drive}")

    # Purge Hugging Face cache
    hf_cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
    if hf_cache_dir.exists():
        shutil.rmtree(hf_cache_dir, ignore_errors=True)
        print("🗑️ Cleared Hugging Face hub cache to free up disk space.\n")


🚀 STARTING PILOT: google/gemma-3-27b-it
[INIT] Model: google/gemma-3-27b-it | Device: cuda | Dtype: torch.bfloat16 | Batch Size: 4 | Run ID: 20261026-222047


Generating (google--gemma-3-27b-it):   0%|          | 0/25 [00:00<?, ?it/s]


[COMPLETE] Saved outputs to /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_google--gemma-3-27b-it.jsonl
[REPORT LOG] Date: 26 Oct | Run ID: 20261026-222047 | Model: google/gemma-3-27b-it | Revision: main | Items: 100
✅ Pilot output saved to Google Drive: /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_google--gemma-3-27b-it.jsonl
🗑️ Cleared Hugging Face hub cache to free up disk space.


🚀 STARTING PILOT: Qwen/Qwen3-32B
[INIT] Model: Qwen/Qwen3-32B | Device: cuda | Dtype: torch.bfloat16 | Batch Size: 8 | Run ID: 20261026-223213


Generating (Qwen--Qwen3-32B):   0%|          | 0/13 [00:00<?, ?it/s]


[COMPLETE] Saved outputs to /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_Qwen--Qwen3-32B.jsonl
[REPORT LOG] Date: 26 Oct | Run ID: 20261026-223213 | Model: Qwen/Qwen3-32B | Revision: main | Items: 100
✅ Pilot output saved to Google Drive: /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_Qwen--Qwen3-32B.jsonl
🗑️ Cleared Hugging Face hub cache to free up disk space.


🚀 STARTING PILOT: allenai/Olmo-3.1-32B-Instruct
[INIT] Model: allenai/Olmo-3.1-32B-Instruct | Device: cuda | Dtype: torch.bfloat16 | Batch Size: 8 | Run ID: 20261026-223958


Generating (allenai--Olmo-3.1-32B-Instruct):   0%|          | 0/13 [00:00<?, ?it/s]


[COMPLETE] Saved outputs to /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_allenai--Olmo-3.1-32B-Instruct.jsonl
[REPORT LOG] Date: 26 Oct | Run ID: 20261026-223958 | Model: allenai/Olmo-3.1-32B-Instruct | Revision: main | Items: 100
✅ Pilot output saved to Google Drive: /content/drive/MyDrive/AI6130 Group Project/results/raw/pilot_allenai--Olmo-3.1-32B-Instruct.jsonl
🗑️ Cleared Hugging Face hub cache to free up disk space.



## 5. Pilot Diagnostics & Ceiling Effect Analysis

In [10]:
import re
import json
import pandas as pd
from IPython.display import display

In [11]:
# Load prompt metadata
meta = {}
with open("data/prompts_pilot.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        meta[row["prompt_id"]] = row

# Process and display diagnostics per model
for model_id in PILOT_MODELS:
    safe_name = model_id.replace("/", "--")
    out_drive = DRIVE_BACKUP_DIR / f"pilot_{safe_name}.jsonl"

    if not out_drive.exists():
        print(f"⚠️ Missing output for {model_id}, skipping diagnostics.")
        continue

    records = []
    with open(out_drive, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            m = meta[r["prompt_id"]]
            text = r["response_text"]
            has_answer_tag = "Answer:" in text

            pred = None
            if has_answer_tag:
                after = text.split("Answer:")[-1].strip()
                if m["task"] == "gsm8k":
                    nums = re.findall(r"-?\d+(?:\.\d+)?", after.replace(",", ""))
                    pred = nums[0] if nums else None
                elif m["task"] != "xstest_safe":
                    lets = re.findall(r"\b([A-E])\b", after)
                    pred = lets[0] if lets else None

            correct = (pred == m["gold"]) if m["gold"] is not None else None
            logits = r.get("logits_meta") or {}

            records.append({
                "prompt_id": r["prompt_id"],
                "task": m["task"],
                "tone": m["tone"],
                "has_answer_tag": has_answer_tag,
                "correct": correct,
                "n_output_tokens": r["n_output_tokens"],
                "refusal_prob": logits.get("first_step_refusal_prob", 0.0),
            })

    df = pd.DataFrame(records)

    print("=" * 70)
    print(f" 📊 DIAGNOSTICS FOR: {model_id}")
    print("=" * 70)

    print("\n=== 1. ACCURACY & CoT LENGTH BY TONE (Graded Tasks, n=80) ===")
    graded_df = df[df["task"] != "xstest_safe"]
    tone_summary = graded_df.groupby("tone").agg(
        format_compliance=("has_answer_tag", "mean"),
        accuracy=("correct", "mean"),
        mean_output_tokens=("n_output_tokens", "mean"),
    ).reindex(["very_rude", "blunt", "neutral", "polite", "very_deferential"])
    display(tone_summary)

    print("\n=== 2. CEILING EFFECT CHECK: ACCURACY BY TASK (Across Tones) ===")
    task_summary = graded_df.groupby(["task", "tone"])["correct"].mean().unstack()
    display(task_summary[["very_rude", "blunt", "neutral", "polite", "very_deferential"]])

    print("\n=== 3. XSTEST SAFE REFUSAL PROPENSITY BY TONE (n=20) ===")
    xst_df = df[df["task"] == "xstest_safe"]
    xst_summary = xst_df.groupby("tone").agg(
        mean_first_step_refusal_prob=("refusal_prob", "mean"),
        max_first_step_refusal_prob=("refusal_prob", "max"),
        mean_output_tokens=("n_output_tokens", "mean"),
    ).reindex(["very_rude", "blunt", "neutral", "polite", "very_deferential"])
    display(xst_summary)
    print("\n\n")

 📊 DIAGNOSTICS FOR: google/gemma-3-27b-it

=== 1. ACCURACY & CoT LENGTH BY TONE (Graded Tasks, n=80) ===


,format_compliance,accuracy,mean_output_tokens
tone,,,
very_rude,1.0,0.8125,150.5000
blunt,1.0,0.8125,48.6875
neutral,1.0,0.8125,140.2500
polite,1.0,0.8125,175.7500
very_deferential,1.0,0.8125,154.1250



=== 2. CEILING EFFECT CHECK: ACCURACY BY TASK (Across Tones) ===


tone,very_rude,blunt,neutral,polite,very_deferential
task,,,,,
arc_challenge,1.0,1.0,1.0,1.0,1.0
commonsense_qa,0.5,0.5,0.5,0.5,0.5
gsm8k,0.75,0.75,0.75,0.75,0.75
openbook_qa,1.0,1.0,1.0,1.0,1.0



=== 3. XSTEST SAFE REFUSAL PROPENSITY BY TONE (n=20) ===


,mean_first_step_refusal_prob,max_first_step_refusal_prob,mean_output_tokens
tone,,,
very_rude,0.0,0.0,456.50
blunt,0.0,0.0,287.75
neutral,0.0,0.0,420.50
polite,0.0,0.0,482.75
very_deferential,0.0,0.0,451.00


... (8 outputs left out)


=== 2. CEILING EFFECT CHECK: ACCURACY BY TASK (Across Tones) ===


tone,very_rude,blunt,neutral,polite,very_deferential
task,,,,,
arc_challenge,1.0,1.0,1.0,1.0,1.0
commonsense_qa,0.5,0.5,0.5,0.25,0.25
gsm8k,0.75,0.75,0.75,0.75,0.75
openbook_qa,0.75,0.75,1.0,0.75,0.75



=== 3. XSTEST SAFE REFUSAL PROPENSITY BY TONE (n=20) ===


,mean_first_step_refusal_prob,max_first_step_refusal_prob,mean_output_tokens
tone,,,
very_rude,0.006625,0.02060,269.75
blunt,0.002207,0.00780,179.75
neutral,0.004542,0.01772,238.75
polite,0.028465,0.09981,288.75
very_deferential,0.147110,0.50063,345.75
